In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
import os

## train

In [2]:
# Định nghĩa các tham số
batch_size = 32
epochs = 20
learning_rate = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Định nghĩa đường dẫn đến tập dữ liệu
data_dir = "data/cic_ddos_2019_images"  # Cập nhật đường dẫn tương ứng
train_dir = os.path.join(data_dir, "train")
valid_dir = os.path.join(data_dir, "valid")
test_dir = os.path.join(data_dir, "test")

In [4]:

# Tiền xử lý dữ liệu
transform = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

In [5]:
# Tạo dataset và dataloader
train_dataset = datasets.ImageFolder(train_dir, transform=transform['train'])
valid_dataset = datasets.ImageFolder(valid_dir, transform=transform['valid'])
test_dataset = datasets.ImageFolder(test_dir, transform=transform['test'])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [6]:
# Load mô hình EfficientNet
model = models.efficientnet_b0(pretrained=True)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, len(train_dataset.classes))
model = model.to(device)

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# Định nghĩa hàm mất mát và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [8]:
# Hàm đánh giá mô hình
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    
    acc = correct / total
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return acc, precision, recall, f1

In [9]:
# Huấn luyện mô hình
def train_model(model, train_loader, valid_loader, criterion, optimizer, epochs):
    best_acc = 0.0
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        train_acc = correct / total
        valid_acc, valid_precision, valid_recall, valid_f1 = evaluate_model(model, valid_loader)
        
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, "
              f"Train Acc: {train_acc:.4f}, Valid Acc: {valid_acc:.4f}, "
              f"Precision: {valid_precision:.4f}, Recall: {valid_recall:.4f}, F1-Score: {valid_f1:.4f}")
        
        if valid_acc > best_acc:
            best_acc = valid_acc
            torch.save(model.state_dict(), "models/EfficientNet.pth")

In [10]:
# Huấn luyện mô hình
train_model(model, train_loader, valid_loader, criterion, optimizer, epochs)

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [1/20], Loss: 1.9306, Train Acc: 0.3704, Valid Acc: 0.1111, Precision: 0.0635, Recall: 0.1111, F1-Score: 0.0694


f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [2/20], Loss: 1.0861, Train Acc: 0.9444, Valid Acc: 0.1667, Precision: 0.0802, Recall: 0.1667, F1-Score: 0.0960


f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Epoch [6/20], Loss: 0.0309, Train Acc: 1.0000, Valid Acc: 0.2778, Precision: 0.3504, Recall: 0.2778, F1-Score: 0.2519


f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [7/20], Loss: 0.0351, Train Acc: 1.0000, Valid Acc: 0.5000, Precision: 0.4877, Recall: 0.5000, F1-Score: 0.4441


f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch [8/20], Loss: 0.0155, Train Acc: 1.0000, Valid Acc: 0.6667, Precision: 0.7540, Recall: 0.6667, F1-Score: 0.6605
Epoch [9/20], Loss: 0.0030, Train Acc: 1.0000, Valid Acc: 0.8333, Precision: 0.8889, Recall: 0.8333, F1-Score: 0.8333
Epoch [10/20], Loss: 0.0025, Train Acc: 1.0000, Valid Acc: 0.7778, Precision: 0.8704, Recall: 0.7778, F1-Score: 0.7852
Epoch [11/20], Loss: 0.0189, Train Acc: 1.0000, Valid Acc: 0.8889, Precision: 0.9259, Recall: 0.8889, F1-Score: 0.8815
Epoch [12/20], Loss: 0.0009, Train Acc: 1.0000, Valid Acc: 0.8889, Precision: 0.9259, Recall: 0.8889, F1-Score: 0.8815
Epoch [13/20], Loss: 0.0023, Train Acc: 1.0000, Valid Acc: 0.8889, Precision: 0.9259, Recall: 0.8889, F1-Score: 0.8815
Epoch [14/20], Loss: 0.0155, Train Acc: 1.0000, Valid Acc: 0.9444, Precision: 0.9630, Recall: 0.9444, F1-Score: 0.9407
Epoch [15/20], Loss: 0.0018, Train Acc: 1.0000, Valid Acc: 0.9444, Precision: 0.9630, Recall: 0.9444, F1-Score: 0.9407
Epoch [16/20], Loss: 0.0054, Train Acc: 1.0000, Va

## đánh giá

In [ ]:
# Đánh giá trên tập test
model.load_state_dict(torch.load("models/EfficientNet.pth"))
test_acc, test_precision, test_recall, test_f1 = evaluate_model(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1-Score: {test_f1:.4f}")


Test Accuracy: 1.0000, Precision: 1.0000, Recall: 1.0000, F1-Score: 1.0000
